In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, BertForSequenceClassification
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

In [ ]:
DATA_PATH = "/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0535/data/clean_news_dataset.csv"

MODEL_DIR = "/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0535/models"
RESULTS_DIR = "/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0535/results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.columns.tolist())

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate content:", df["content"].duplicated().sum())

In [ ]:
df["clean_text"] = df["clean_text"].fillna("")

print("Missing clean_text values:", df["clean_text"].isnull().sum())

In [ ]:
X = df["clean_text"]
y = df["label"]

print("Features:", X.shape)
print("Target:", y.shape)
print("Classes:", sorted(y.unique()))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
MAX_LENGTH = 256

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)

print("Train tokens:", len(train_encodings["input_ids"]))
print("Test tokens:", len(test_encodings["input_ids"]))

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

In [ ]:
BATCH_SIZE = 16

train_dataset = NewsDataset(train_encodings, y_train)
test_dataset = NewsDataset(test_encodings, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("BERT model loaded successfully!")

In [ ]:
LEARNING_RATE = 2e-5
EPOCHS = 2

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Optimizer created successfully!")

In [ ]:
model.train()

for epoch in range(EPOCHS):
    total_loss = 0

    for batch in train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - Loss: {average_loss:.4f}"
    )

In [ ]:
model.eval()

y_pred = []
y_prob = []

with torch.no_grad():
    for batch in test_loader:
        labels = batch["labels"].to(device)

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)

        predictions = torch.argmax(probabilities, dim=1)

        y_pred.extend(predictions.cpu().numpy())
        y_prob.extend(probabilities[:, 1].cpu().numpy())

print("Predictions generated:", len(y_pred))

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", round(accuracy * 100, 2))

In [ ]:
precision = precision_score(y_test, y_pred)

print("Precision:", precision)
print("Precision (%):", round(precision * 100, 2))

In [ ]:
recall = recall_score(y_test, y_pred)

print("Recall:", recall)
print("Recall (%):", round(recall * 100, 2))

In [ ]:
f1 = f1_score(y_test, y_pred)

print("F1 Score:", f1)
print("F1 Score (%):", round(f1 * 100, 2))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fake", "True"]
)

disp.plot()
plt.title("BERT Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Fake", "True"]
))

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)
print("ROC-AUC (%):", round(roc_auc * 100, 2))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"BERT (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("BERT ROC Curve")
plt.legend()
plt.show()

In [ ]:
BERT_MODEL_PATH = os.path.join(MODEL_DIR, "bert_model")

model.save_pretrained(BERT_MODEL_PATH)
tokenizer.save_pretrained(BERT_MODEL_PATH)

print("BERT model saved successfully!")

In [ ]:
results = pd.DataFrame({
    "Model": ["BERT"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1": [f1],
    "ROC_AUC": [roc_auc]
})

RESULTS_PATH = os.path.join(RESULTS_DIR, "bert_results.csv")

results.to_csv(
    RESULTS_PATH,
    index=False
)

display(results)
print("Results saved successfully!")

In [ ]:
loaded_model = BertForSequenceClassification.from_pretrained(
    BERT_MODEL_PATH
)

loaded_model.to(device)
loaded_model.eval()

print("BERT model reloaded successfully!")
print("Device:", device)

In [ ]:
sample_text = X_test.iloc[0]

sample_encoding = tokenizer(
    sample_text,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

sample_encoding = {
    key: value.to(device)
    for key, value in sample_encoding.items()
}

with torch.no_grad():
    sample_output = loaded_model(**sample_encoding)
    sample_probability = torch.softmax(
        sample_output.logits,
        dim=1
    )[0, 1].item()

sample_prediction = int(sample_probability >= 0.5)

print("Actual Label:", y_test.iloc[0])
print("Predicted Label:", sample_prediction)
print("Prediction Probability:", sample_probability)
print("Model verification successful!")

In [ ]:
print("Final Dataset Shape:", df.shape)
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))
print("Vocabulary Size:", tokenizer.vocab_size)
print("Maximum Sequence Length:", MAX_LENGTH)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision:", round(precision * 100, 2))
print("Recall:", round(recall * 100, 2))
print("F1 Score:", round(f1 * 100, 2))
print("ROC-AUC:", round(roc_auc * 100, 2))
print("Model Path:", BERT_MODEL_PATH)
print("Results Path:", RESULTS_PATH)